# Mission d'adaptation : un nouveau site, sans code fourni
**ÉNONCÉ**

Vous savez maintenant scraper et crawler un site (Books to Scrape), et le faire poliment. **Cette mission change de site, et on ne vous donne aucun code de collecte.** C'est à vous d'adapter votre démarche à une structure différente.

**Le site :** `quotes.toscrape.com`, des citations célèbres, créé lui aussi pour s'entraîner.

## Ce que vous devez produire
1. **`citations`** : un tableau de **toutes** les citations (toutes les pages), avec les colonnes `texte`, `auteur`, `tags`, `url_page`, `url_auteur`.
2. **`auteurs`** : un tableau de la fiche de **chaque auteur, une seule fois**, avec `url_auteur`, `auteur`, `date_naissance`, `lieu_naissance`, `biographie`.
3. **`nb_requetes`** : le nombre de pages que votre script a demandées (pages de citations + fiches d'auteurs, sans compter `robots.txt`).
4. **`citations_completes`** : les citations enrichies de la fiche de leur auteur.
5. **`top_auteurs`** et **`top_tags`** : les 5 auteurs les plus cités et les 10 tags les plus fréquents.
6. **Une note de 5 lignes** : ce qui change par rapport à Books to Scrape, et comment vous l'avez géré.

## Règles
Les règles du notebook précédent s'appliquent : vous vous **présentez** (User-Agent avec contact), vous **lisez le robots.txt**, vous faites une **pause**. Ces outils vous sont fournis plus bas.

## Pour vous mettre sur la piste (sans code)
- **Inspectez** le site dans votre navigateur avant d'écrire quoi que ce soit. Qu'est-ce qui est pareil ? Qu'est-ce qui est différent ?
- Regardez **la forme des adresses** des liens « Next » et « (about) ».
- Un **même auteur** apparaît dans de nombreuses citations. Faut-il télécharger sa fiche à chaque fois ?
- Une citation a **plusieurs tags**. Comment stocker cela dans une seule case ?
- Le texte d'une citation est entouré de **guillemets typographiques**. Faut-il les garder ?

In [1]:
import re, time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

SITE = "https://quotes.toscrape.com"      # un NOUVEAU site-école, à la structure différente de Books to Scrape

def controle(condition, ok, ko):
    print("✅", ok) if condition else print("❌", ko)

## Étape 1 — Se présenter
Comme dans le notebook précédent : remplacez `___` par votre contact (adresse e-mail ou adresse d'une page de projet).

In [2]:
ENTETES = {"User-Agent": "TPFormation/1.0 (https://github.com/GregoryElBajoury) python-requests"}   # TODO : remplacez ___ par votre contact
print(ENTETES)

{'User-Agent': 'TPFormation/1.0 (https://github.com/GregoryElBajoury) python-requests'}


In [3]:
# Contrôle (ne pas modifier)
ua = ENTETES['User-Agent']
controle(('@' in ua or 'http' in ua) and '___' not in ua and not any(m in ua.lower() for m in ('example', 'exemple', 'votre.')), 'User-Agent renseigné : ' + ua, 'Remplacez ___ par un vrai contact')

✅ User-Agent renseigné : TPFormation/1.0 (https://github.com/GregoryElBajoury) python-requests


## Étape 2 — Vos outils (fournis)
La fonction de requête polie et la lecture du `robots.txt`, issues du notebook précédent.

In [4]:
PAUSE = 1.0

def get_poli(url, essais=3):
    for n in range(essais):
        time.sleep(PAUSE)
        reponse = requests.get(url, headers=ENTETES, timeout=20)
        if reponse.status_code in (429, 503):
            time.sleep(2 ** (n + 1))
            continue
        return reponse
    raise RuntimeError("Échec après " + str(essais) + " essais : " + url)

def lire_robots(base):
    reponse = requests.get(base + "/robots.txt", headers=ENTETES)
    print(base, "-> code", reponse.status_code)
    if reponse.status_code == 404:
        print("   pas de fichier robots.txt : aucune restriction annoncée")
        return None
    if reponse.status_code != 200:
        raise RuntimeError("Réponse inattendue pour robots.txt : on s'abstient par prudence")
    regles = RobotFileParser(); regles.parse(reponse.text.splitlines())
    print(reponse.text[:500])
    return regles

## Étape 3 — Consulter le robots.txt
Lisez le `robots.txt` de ce site, puis demandez s'il est permis de visiter la page 2 des citations (`/page/2/`) et la fiche d'un auteur (`/author/Albert-Einstein`). Stockez les deux réponses (vrai ou faux) dans `autorise_pages` et `autorise_auteur`.

In [5]:
regles = lire_robots(SITE)
# TODO : calculer autorise_pages et autorise_auteur (si regles vaut None, aucune restriction annoncée)
if regles is None:
    autorise_pages = True
    autorise_auteur = True
else:
    autorise_pages = regles.can_fetch(ENTETES["User-Agent"], "/page/2/")
    autorise_auteur = regles.can_fetch(ENTETES["User-Agent"], "/author/Albert-Einstein")

print("Autorisé pour /page/2/ :", autorise_pages)
print("Autorisé pour /author/Albert-Einstein :", autorise_auteur)

https://quotes.toscrape.com -> code 404
   pas de fichier robots.txt : aucune restriction annoncée
Autorisé pour /page/2/ : True
Autorisé pour /author/Albert-Einstein : True


In [6]:
# Contrôle (ne pas modifier)
controle(isinstance(autorise_pages, bool) and isinstance(autorise_auteur, bool), 'Autorisations établies : pages=%s, auteur=%s' % (autorise_pages, autorise_auteur), 'Attendu : deux booléens')

✅ Autorisations établies : pages=True, auteur=True


## Étape 4 — Collecter toutes les citations *(à écrire seul)*
Objectif : la variable **`citations`** (un `DataFrame`) et le compteur **`nb_requetes`**. Cette cellule est **vide** : à vous d'écrire la boucle.

In [8]:
# À vous : collecter toutes les citations (variables attendues : citations, nb_requetes)

def collecter_citations(site_url: str) -> tuple[pd.DataFrame, int]:
    
    nb_requetes: int = 0
    url: str | None = urljoin(site_url, "/")
    liste_citations: list[dict] = []
    
    while url:
        reponse = get_poli(url)
        nb_requetes += 1
        
        if reponse.status_code != 200:
            print(f" Attention : code {reponse.status_code} reçu pour l'URL {url}")
            break
            
        page = BeautifulSoup(reponse.content, "html.parser")
        
        for bloc in page.select("div.quote"):
            # Sécurité : on vérifie que les éléments clés existent
            texte_elem = bloc.select_one("span.text")
            auteur_elem = bloc.select_one("small.author")
            lien_elem = bloc.select_one("span a")
            
            if not (texte_elem and auteur_elem and lien_elem):
                continue
                
            texte: str = texte_elem.get_text().strip().strip("“„”\"")
            auteur: str = auteur_elem.get_text().strip()
            url_auteur: str = urljoin(site_url, lien_elem.get("href", ""))
            tags: list[str] = [tag.get_text().strip() for tag in bloc.select("div.tags a.tag")]
            
            liste_citations.append({
                "texte": texte,
                "auteur": auteur,
                "tags": tags,
                "url_page": url,
                "url_auteur": url_auteur
            })
            
        # Utilisation de .get() pour éviter un TypeError si le lien 'next' change ou est mal formé
        suivant = page.select_one("li.next a")
        url = urljoin(site_url, suivant.get("href")) if suivant and suivant.get("href") else None
        
    citations_df = pd.DataFrame(liste_citations)
    print(f" Collecte terminée : {len(citations_df)} citations récupérées en {nb_requetes} requêtes.")
    return citations_df, nb_requetes

citations, nb_requetes = collecter_citations(SITE)




 Collecte terminée : 100 citations récupérées en 10 requêtes.


In [9]:
# Contrôle (ne pas modifier)
colonnes = {"texte", "auteur", "tags", "url_page", "url_auteur"}
controle(isinstance(citations, pd.DataFrame) and colonnes <= set(citations.columns) and len(citations) >= 10,
         "%d citations collectées, colonnes correctes" % len(citations), "Attendu : un DataFrame avec les colonnes " + ", ".join(sorted(colonnes)) + " et au moins 10 lignes")
controle(citations["tags"].map(lambda t: isinstance(t, list)).all() and not citations["texte"].str.startswith(("“", '"')).any(),
         "Les tags sont des listes et les guillemets typographiques sont retirés", "Les tags doivent être des listes, et le texte ne doit plus commencer par un guillemet")
controle(citations["url_auteur"].str.startswith("http").all() and citations["url_page"].str.startswith("http").all(),
         "Toutes les adresses sont complètes", "Les adresses doivent commencer par http : pensez aux liens relatifs")
controle(not citations.duplicated(subset=["texte", "auteur"]).any(), "Aucune citation en double", "Il y a des citations en double : une page a-t-elle été lue deux fois ?")

✅ 100 citations collectées, colonnes correctes
✅ Les tags sont des listes et les guillemets typographiques sont retirés
✅ Toutes les adresses sont complètes
✅ Aucune citation en double


## Étape 5 — Collecter la fiche de chaque auteur *(à écrire seul)*
Objectif : la variable **`auteurs`**, avec **une seule ligne par auteur**. Inspectez une fiche d'auteur pour trouver où sont la date et le lieu de naissance, et la biographie. **Attention au nombre de requêtes** : le contrôle suivant le vérifie.

In [10]:
def collecter_auteurs(urls_uniques: list[str], nb_req_actuel: int) -> tuple[pd.DataFrame, int]:
    
    liste_auteurs: list[dict] = []
    nb_requetes: int = nb_req_actuel
    
    for url_a in urls_uniques:
        reponse = get_poli(url_a)
        nb_requetes += 1
        
        if reponse.status_code != 200:
            print(f" Attention : code {reponse.status_code} reçu pour l'URL {url_a}")
            continue
            
        page = BeautifulSoup(reponse.content, "html.parser")
        
        # Sécurité : on vérifie que les éléments de la fiche existent - voir si possible filtre aut
        nom_elem = page.select_one("h3.author-title")
        date_elem = page.select_one("span.author-born-date")
        lieu_elem = page.select_one("span.author-born-location")
        bio_elem = page.select_one("div.author-description")
        
        if not (nom_elem and date_elem and lieu_elem and bio_elem):
            continue
            
        nom: str = nom_elem.get_text().strip()
        date_naissance: str = date_elem.get_text().strip()
        lieu_naissance: str = lieu_elem.get_text().strip()
        biographie: str = bio_elem.get_text().strip()
        
        liste_auteurs.append({
            "url_auteur": url_a,
            "auteur": nom,
            "date_naissance": date_naissance,
            "lieu_naissance": lieu_naissance,
            "biographie": biographie
        })
        
    auteurs_df = pd.DataFrame(liste_auteurs)
    print(f" Collecte des fiches auteurs terminée : {len(auteurs_df)} fiches uniques récupérées.")
    print(f"Nombre total de requêtes effectuées : {nb_requetes}")
    return auteurs_df, nb_requetes

auteurs, nb_requetes = collecter_auteurs(citations["url_auteur"].unique(), nb_requetes)

 Collecte des fiches auteurs terminée : 50 fiches uniques récupérées.
Nombre total de requêtes effectuées : 60


In [11]:
# Contrôle (ne pas modifier)
colonnes_a = {"url_auteur", "auteur", "date_naissance", "lieu_naissance", "biographie"}
controle(isinstance(auteurs, pd.DataFrame) and colonnes_a <= set(auteurs.columns) and auteurs["url_auteur"].is_unique,
         "%d fiches d'auteurs, une seule par auteur" % len(auteurs), "Attendu : les colonnes " + ", ".join(sorted(colonnes_a)) + " et une seule ligne par auteur")
controle(set(citations["url_auteur"]) <= set(auteurs["url_auteur"]), "Tous les auteurs cités ont une fiche", "Certains auteurs cités n'ont pas de fiche")
attendu = citations["url_page"].nunique() + citations["url_auteur"].nunique()
controle(nb_requetes <= attendu + 2, "%d requêtes pour %d attendues au minimum : fiches non redemandées" % (nb_requetes, attendu), "%d requêtes alors que %d suffisent : un auteur a été redemandé plusieurs fois" % (nb_requetes, attendu))

✅ 50 fiches d'auteurs, une seule par auteur
✅ Tous les auteurs cités ont une fiche
✅ 60 requêtes pour 60 attendues au minimum : fiches non redemandées


## Étape 6 — Fusionner et analyser *(à écrire seul)*
Objectifs : **`citations_completes`** (chaque citation avec la fiche de son auteur), **`top_auteurs`** (les 5 auteurs les plus cités) et **`top_tags`** (les 10 tags les plus fréquents). Indice : pour compter des tags stockés dans des listes, il faut d'abord « déplier » ces listes.

In [12]:
# À vous : citations_completes, top_auteurs, top_tags

def fusionner_et_analyser(citations_df: pd.DataFrame, auteurs_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    """
    Fusionne les citations avec les fiches auteurs et calcule les classements (tops).
    Retourne le DataFrame complet, le top des auteurs et le top des tags.
    """
    # Fusion des citations et des auteurs
    citations_completes: pd.DataFrame = pd.merge(citations_df, auteurs_df, on="url_auteur", how="left")
    
    # Top 5 des auteurs les plus cités
    top_auteurs: pd.Series = citations_df["auteur"].value_counts().head(5)
    
    # Top 10 des tags les plus fréquents
    top_tags: pd.Series = citations_df.explode("tags")["tags"].value_counts().head(10)
    
    print(" Fusion et classements calculés avec succès !")
    return citations_completes, top_auteurs, top_tags

citations_completes, top_auteurs, top_tags = fusionner_et_analyser(citations, auteurs)




 Fusion et classements calculés avec succès !


In [13]:
# Les verifs de JeanMich
print("\n--- TOP 5 AUTEURS ---")
print(top_auteurs)

print("\n--- TOP 10 TAGS ---")
print(top_tags)



--- TOP 5 AUTEURS ---
auteur
Albert Einstein    10
J.K. Rowling        9
Marilyn Monroe      7
Dr. Seuss           6
Mark Twain          6
Name: count, dtype: int64

--- TOP 10 TAGS ---
tags
love             14
inspirational    13
life             13
humor            12
books            11
reading           7
friendship        5
friends           4
truth             4
simile            3
Name: count, dtype: int64


In [14]:
# Contrôle (ne pas modifier)
controle(len(citations_completes) == len(citations) and "date_naissance" in citations_completes.columns and citations_completes["date_naissance"].notna().all(),
         "Fusion correcte : chaque citation a la fiche de son auteur", "La fusion doit garder toutes les citations, chacune avec la date de naissance de son auteur")
controle(len(top_auteurs) == min(5, citations["auteur"].nunique()) and len(top_tags) == min(10, citations.explode("tags")["tags"].nunique()) and top_auteurs.is_monotonic_decreasing and top_tags.is_monotonic_decreasing,
         "Classements corrects : 5 auteurs et 10 tags, du plus fréquent au moins fréquent", "Attendu : 5 auteurs et 10 tags triés par fréquence décroissante")

✅ Fusion correcte : chaque citation a la fiche de son auteur
✅ Classements corrects : 5 auteurs et 10 tags, du plus fréquent au moins fréquent


## Étape 7 — Votre note (5 lignes)
Répondez ici, en 5 lignes maximum :

1. Ce qui était **identique** à Books to Scrape.
2. Ce qui était **différent**.
3. Ce que vous avez dû **changer** dans votre démarche.
4. Comment vous avez évité de **redemander** les fiches d'auteurs.
5. Ce qui, à votre avis, **casserait** votre script si le site changeait.

> 1. La structure globale du crawling par page et l'utilisation de BeautifulSoup étaient identiques à Books to Scrape.
>2. La différence venait de la relation de données, orientée ici entre des citations et les fiches détaillées de leurs auteurs respectifs.
>3. J'ai dû adapter la boucle de collecte pour extraire des listes de tags et cibler les pages de profils d'auteurs individuels.
>4. L'utilisation d'une liste d'URLs uniques via `.unique()` a permis d'éviter de redemander inutilement la fiche d'un même auteur.
>5. Toute modification future de la structure HTML, des classes CSS ou l'ajout d'une barrière anti-bot casserait le script.


Du coup on pourrait peut etre explorer certaines possibilités :
> - Pour contrer les changements de structure HTML/CSS : Privilégie les sélecteurs sémantiques ou des attributs stables plutôt que des classes CSS fragiles, et mets en place des contrôles de qualité des données en amont pour détecter immédiatement si un champ change ou disparaît.

>- Pour contrer les barrières anti-bot : Introduis des délais aléatoires (jitter) entre tes requêtes pour imiter un comportement humain, fais tourner tes adresses IP via un pool de proxys, ou utilise des navigateurs automatisés (comme Playwright) si le site déploie de lourdes sécurités.